# 04 — Business Insights
**E-Commerce Sales Analysis**

Deep dive into customer value — RFM segmentation, Pareto analysis, and actionable business recommendations.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# ── Paths ──────────────────────────────────────────────
ROOT        = os.path.abspath("..")
CLEAN_PATH  = os.path.join(ROOT, "data", "processed", "cleaned_data.csv")
FIGURES_DIR = os.path.join(ROOT, "reports", "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

# ── Style ──────────────────────────────────────────────
plt.rcParams.update({
    "figure.figsize"   : (12, 5),
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "axes.grid"        : True,
    "grid.alpha"       : 0.3,
    "font.size"        : 11,
})

print("Setup complete")

In [ ]:
# ── Load cleaned data ──────────────────────────────────
df = pd.read_csv(CLEAN_PATH, parse_dates=["InvoiceDate"])
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

## 1 — Pareto Analysis (80/20 Rule)

In [ ]:
product_revenue = (
    df.groupby('Description')['TotalPrice']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

product_revenue['cumulative_pct'] = (
    product_revenue['TotalPrice'].cumsum() /
    product_revenue['TotalPrice'].sum() * 100
)
product_revenue['product_pct'] = (
    np.arange(1, len(product_revenue) + 1) / len(product_revenue) * 100
)

# How many products = 80% of revenue?
pareto_threshold = product_revenue[product_revenue['cumulative_pct'] <= 80]
print(f"Products generating 80% of revenue: {len(pareto_threshold):,} "
      f"({len(pareto_threshold)/len(product_revenue)*100:.1f}% of all products)")

fig, ax = plt.subplots()
ax.plot(product_revenue['product_pct'], product_revenue['cumulative_pct'],
        linewidth=2, color='#2563eb')
ax.axhline(80, color='#ef4444', linestyle='--', linewidth=1.5, label='80% revenue')
ax.axvline(len(pareto_threshold)/len(product_revenue)*100,
           color='#f97316', linestyle='--', linewidth=1.5, label=f'{len(pareto_threshold)/len(product_revenue)*100:.0f}% of products')
ax.set_title('Pareto Analysis — Products vs Revenue', fontsize=14, fontweight='bold')
ax.set_xlabel('% of Products')
ax.set_ylabel('Cumulative % of Revenue')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'pareto_analysis.png'), dpi=150)
plt.show()
print("Saved: pareto_analysis.png")

## 2 — Top 10 Customers by Revenue

In [ ]:
top_customers = (
    df.groupby('CustomerID')['TotalPrice']
    .sum()
    .sort_values(ascending=True)
    .tail(10)
)

fig, ax = plt.subplots()
bars = ax.barh(top_customers.index.astype(str), top_customers.values, color='#2563eb')
ax.bar_label(bars, fmt='£{:,.0f}', padding=5, fontsize=9)
ax.set_title('Top 10 Customers by Revenue', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Revenue (£)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'top_customers.png'), dpi=150)
plt.show()
print("Saved: top_customers.png")

## 3 — RFM Customer Segmentation

**R**ecency — how recently did the customer buy?  
**F**requency — how often do they buy?  
**M**onetary — how much do they spend?

In [ ]:
# Reference date = 1 day after last invoice
ref_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate', lambda x: (ref_date - x.max()).days),
    Frequency = ('InvoiceNo',   'nunique'),
    Monetary  = ('TotalPrice',  'sum')
).reset_index()

print(f"RFM table: {len(rfm):,} customers")
rfm.head()

In [ ]:
# ── Score each dimension 1–4 ───────────────────────────
rfm['R_score'] = pd.qcut(rfm['Recency'],   q=4, labels=[4, 3, 2, 1]).astype(int)
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=4, labels=[1, 2, 3, 4]).astype(int)
rfm['M_score'] = pd.qcut(rfm['Monetary'],  q=4, labels=[1, 2, 3, 4]).astype(int)

rfm['RFM_Score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

# ── Assign segments ────────────────────────────────────
def segment(score):
    if score >= 10: return '💎 Champions'
    elif score >= 8: return '🌟 Loyal'
    elif score >= 6: return '🔄 Potential'
    elif score >= 4: return '⚠️ At Risk'
    else:            return '❌ Lost'

rfm['Segment'] = rfm['RFM_Score'].apply(segment)

print(rfm['Segment'].value_counts())

In [ ]:
# ── RFM Segment chart ──────────────────────────────────
segment_counts = rfm['Segment'].value_counts()
colors = ['#2563eb', '#3b82f6', '#60a5fa', '#93c5fd', '#bfdbfe']

fig, ax = plt.subplots()
bars = ax.barh(segment_counts.index, segment_counts.values, color=colors[:len(segment_counts)])
ax.bar_label(bars, fmt='%d customers', padding=5, fontsize=9)
ax.set_title('Customer Segments (RFM)', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Customers')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'rfm_segments.png'), dpi=150)
plt.show()
print("Saved: rfm_segments.png")

In [ ]:
# ── Segment revenue share ──────────────────────────────
segment_revenue = rfm.groupby('Segment')['Monetary'].sum().sort_values(ascending=True)

fig, ax = plt.subplots()
bars = ax.barh(segment_revenue.index, segment_revenue.values, color='#2563eb')
ax.bar_label(bars, fmt='£{:,.0f}', padding=5, fontsize=9)
ax.set_title('Revenue by Customer Segment', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Revenue (£)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'segment_revenue.png'), dpi=150)
plt.show()
print("Saved: segment_revenue.png")

## 📝 Business Insights Summary

| Insight | Finding | Recommendation |
|---|---|---|
| Pareto | ~20% of products = ~80% of revenue | Focus stock and marketing on top products |
| Top customers | Small group drives large revenue share | Build loyalty program for Champions |
| At Risk segment | Customers who used to buy but stopped | Re-engagement email campaign |
| Lost segment | Churned customers | Analyse why and prevent with other segments |

---

**All notebooks complete ✅**  
Next step → `app/dashboard.py`